In [184]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import pandas as pd
import numpy as np
c = pd.read_csv("../data/comments_raw.csv")
brands = pd.read_csv("../data/brands.csv")

In [185]:
# Rule: on a column, string methods always go through .str. 
# c["brand_raw"].lower() gives an AttributeError, because a Series has no .lower().

In [186]:
s = pd.Series(["ABC", "BCD"])
s.str.lower()       # "GLOWLY" → "glowly"
s.str.upper()       # "glowly" → "GLOWLY"
s.str.title()       # "glowly" → "Glowly"
s.str.strip()       # " glowly " → "glowly"   (removes spaces at both ends)

0    abc
1    bcd
dtype: str

0    ABC
1    BCD
dtype: str

0    Abc
1    Bcd
dtype: str

0    ABC
1    BCD
dtype: str

In [187]:
s.str.strip().str.lower()     # ✓
# s.str.strip().lower()         # ✗ AttributeError

0    abc
1    bcd
dtype: str

In [188]:
s.str.len()               # number of characters (spaces count)
s.str.contains("refund")  # True/False: is "refund" anywhere inside?
s.str.startswith("@")     # True/False: does it start with "@"?
s.str.endswith("?")       # True/False: does it end with "?"
s.str.count("#")          # how many times "#" appears

0    3
1    3
dtype: int64

0    False
1    False
dtype: bool

0    False
1    False
dtype: bool

0    False
1    False
dtype: bool

0    0
1    0
dtype: int64

In [189]:
s.str.contains("refund", case=False)        # ignore upper/lower case
s.str.contains("refund", na=False)          # treat missing text as False
s.str.contains("refund|delivery", case=False)   # | inside the string means OR

# Make na=False a habit. In your pandas version missing text already gives False, but in older pandas it gives NaN, and a mask with NaN crashes when you filter with it.

0    False
1    False
dtype: bool

0    False
1    False
dtype: bool

0    False
1    False
dtype: bool

In [190]:
s.str[0]        # first character of each value
s.str[1:]       # everything after the first character  ("@hana" → "hana")
s.str[:3]       # first 3 characters

# split cuts each string into a list, and then .str[n] picks an item from each list:

0    A
1    B
dtype: str

0    BC
1    CD
dtype: str

0    ABC
1    BCD
dtype: str

In [191]:
names = pd.Series(["red apple", "green pear"])
names.str.split(" ")          # [["red", "apple"], ["green", "pear"]]
names.str.split(" ").str[1]   # ["apple", "pear"]

0     [red, apple]
1    [green, pear]
dtype: object

0    apple
1     pear
dtype: object

.str.replace("a", "b")	inside each string (part of the text)	"$12.99" → remove "$" → "12.99"
.replace("a", "b")	the whole value, which must match exactly	"snack box" → "snackbox" (only if the cell is exactly "snack box")

In [192]:
s = pd.Series(["apple", "apple pie", "pineapple"])
s.str.replace("apple", "X")    # ["X", "X pie", "pineX"]      ← every occurrence inside
s.replace("apple", "X")        # ["X", "apple pie", "pineapple"]  ← only the exact match

# Analogy: .str.replace is a word processor’s “find & replace” that edits inside sentences. .replace swaps whole labels, like changing a name tag.

# .replace can take a dictionary to map many values at once, which is perfect for fixing category names:
s.replace({"glowly": "Glowly", "snackbox": "SnackBox"})

0        X
1    X pie
2    pineX
dtype: str

0            X
1    apple pie
2    pineapple
dtype: str

0        apple
1    apple pie
2    pineapple
dtype: str

In [193]:
prices = pd.Series(["$5", "$1,200"])
clean = prices.str.replace("$", "").str.replace(",", "").astype(float)
clean.sum()     # 1205.0

# read_csv turns "N/A", "NA" and empty cells into NaN automatically, so there’s no need to clean those.

np.float64(1205.0)

In [194]:
c["brand"] = (c["brand_raw"]
              .str.strip()
              .str.lower()
              .replace({"snack box": "snackbox"}))
c["brand_raw"].nunique()   # 15 messy versions
c["brand"].nunique()       # should be 5

15

5

## 2.9 Cheat sheet: Text with .str

| I want… | Write |
|---|---|
| lowercase / uppercase / Title | `s.str.lower()` / `.str.upper()` / `.str.title()` |
| remove spaces at the ends | `s.str.strip()` |
| several steps | `s.str.strip().str.lower()` (`.str` every time) |
| length | `s.str.len()` |
| contains word (mask) | `s.str.contains("x", case=False, na=False)` |
| contains A or B | `s.str.contains("a\|b", case=False, na=False)` |
| starts / ends with | `s.str.startswith("@")` / `s.str.endswith("?")` |
| count a character | `s.str.count("#")` |
| slice each value | `s.str[1:]` |
| split, take part | `s.str.split(" ").str[0]` |
| replace part of the text | `s.str.replace("$", "")` |
| replace whole values | `s.replace({"old": "new"})` |
| text → number | `.str.replace(...)`, then `.astype(float)` |

**Rules**
- On a column, string methods need `.str` (every step)
- `.str.replace` = inside the text; `.replace` = whole value
- Check `nunique()` before and after cleaning
- `contains` with `na=False` so the mask has no NaN

In [195]:
s = pd.Series(["  Apple", "banana ", "APPLE pie", None, "apple"])

In [196]:
s.str.strip().tolist() #["Apple", "banana", "APPLE pie", NaN, "apple"]
s.str.lower().str.strip().tolist() #["apple", "banana", "apple pie", NaN, "apple"]
s.str.len().tolist() #(count the spaces) [7, 7, 9, NaN, 5]
s.str.contains("apple", na=False).sum() #1
s.str.contains("apple", case=False, na=False).sum() #3
s.replace("apple", "X").tolist() #["  Apple", "banana ", "APPLE pie", NaN, "X"]
s.str.lower().str.replace("apple", "X").tolist() #["  X", "banana ", "X pie", NaN, "X"]
s.str.lower().replace("apple", "X").tolist() #["  apple", "banana ", "apple pie", NaN, "X"]
pd.Series(["red apple", "green pear"]).str.split(" ").str[1].tolist() #['apple', 'pear']
# s.str.strip().lower()  #what happens and why? AttributeError-- no .str before lower()

['Apple', 'banana', 'APPLE pie', nan, 'apple']

['apple', 'banana', 'apple pie', nan, 'apple']

[7.0, 7.0, 9.0, nan, 5.0]

np.int64(1)

np.int64(3)

['  Apple', 'banana ', 'APPLE pie', nan, 'X']

['  X', 'banana ', 'X pie', nan, 'X']

['  apple', 'banana ', 'apple pie', nan, 'X']

['apple', 'pear']

In [197]:
# B1. Look at the mess

# (a) Show c.head(10).
c.head(10)
# (b) How many unique values does brand_raw have?
c['brand_raw'].nunique()
# (c) Show them with .unique().
c['brand_raw'].unique()
# (d) Comment: list the 3 different kinds of mess you can see.
#case, spaces in between text, trailing spaces

,comment_id,post_id,user,brand_raw,text,price_raw,brand
0,C001,P0037,@gabi_x,GLOWLY,Is this vegan?,£8,glowly
1,C002,P0066,@hana,URBANTHREAD,LOVE the new flavour #snacktime,NaN,urbanthread
2,C003,P0137,@ben.t,FITFUEL,Refund please. Wrong size.,£8,fitfuel
3,C004,P0024,@gabi_x,snackbox,Price is too high tbh,NaN,snackbox
4,C005,P0088,@finn,GLOWLY,NaN,NaN,glowly
5,C006,P0004,@hana,HOMENEST,love love love,NaN,homenest
6,C007,P0175,@jo.l,snackbox,meh,NaN,snackbox
7,C008,P0172,@ben.t,GLOWLY,Love it #fitfam,NaN,glowly
8,C009,P0135,@chloe99,FitFuel,how much is this?,NaN,fitfuel
9,C010,P0015,@hana,Glowly,how much is this?,NaN,glowly


15

<ArrowStringArray>
[     'GLOWLY', 'URBANTHREAD',    'FITFUEL ',    'snackbox',    'HOMENEST',
     'FitFuel',     'Glowly ',      'glowly',   'Snack Box',    'homenest',
   'HomeNest ',     ' glowly',     'fitfuel',   ' SnackBox',   'SNACKBOX ']
Length: 15, dtype: str

In [198]:
# B2. Clean brand names: step 1

# (a) Make brand = brand_raw stripped and lowercased.
c['brand']= c['brand_raw'].str.strip().str.lower()
# (b) nunique() of brand.
c['brand'].nunique()
# (c) Show .unique().
c['brand'].unique()
# (d) Comment: one problem is left. Which one, and why couldn’t strip/lower fix it?
#snackbox space in between snack box

6

<ArrowStringArray>
['glowly', 'urbanthread', 'fitfuel', 'snackbox', 'homenest', 'snack box']
Length: 6, dtype: str

In [199]:
# B3. Clean brand names: step 2

# (a) Fix the leftover problem with .replace.
c['brand']= c['brand'].replace("snack box", "snackbox")
c['brand'].unique()
# (b) Map all 5 lowercase names to the official spelling in brands.csv ("Glowly", "SnackBox", "FitFuel", "UrbanThread", "HomeNest") using .replace with a dictionary.
c['brand']= c['brand'].replace(
    {"glowly": "Glowly", "snackbox": "SnackBox", "fitfuel": "FitFuel", "urbanthread": "UrbanThread", "homenest": "HomeNest"}
)
# (c) nunique() should be 5.
c['brand'].nunique()
# (d) Comments per brand with value_counts().
c['brand'].value_counts()
# (e) Comment: why not just use .str.title()? (Try it on "snackbox".)
#.title would make it Snackbox--this doesn't match the original

<ArrowStringArray>
['glowly', 'urbanthread', 'fitfuel', 'snackbox', 'homenest']
Length: 5, dtype: str

5

brand
Glowly         13
HomeNest       11
SnackBox        8
FitFuel         7
UrbanThread     1
Name: count, dtype: int64

In [200]:
# B4. Prove it merges

# (a) Left merge c with brands on brand (2.7).
c_m= c.merge(brands, on='brand', how='left')
# (b) Count NaN in category.
c_m['category'].isna().sum()
# (c) Comment: which brand are those NaN rows, and why? (Hint: 2.7)
#HomeNest-- missing in brands df

np.int64(11)

In [201]:
# B5. Missing text

# (a) How many comments have no text?
c_m['text'].isna().sum()
# (b) Make text_len = length of text.
c_m['text_len'] = c_m['text'].str.len()
# (c) Mean text_len, rounded to 1.
c_m['text_len'].mean().round(1)
# (d) Comment: did the missing texts count as 0 in the mean? (Hint: 2.3)
#no, they get skipped 

np.int64(6)

np.float64(20.8)

In [202]:
# B6. Searching text

# (a) How many comments mention "refund" (any case)?
c_m['text'].str.contains('refund', case=False).sum()
# (b) How many mention "love" case-sensitive?
c_m['text'].str.contains('love').sum()
# (c) How many mention "love" any case?
c_m['text'].str.contains('love', case=False).sum()
# (d) Comment: why are (b) and (c) different?
#case sensitivity
# (e) How many comments are questions (end with "?")?
c_m['text'].str.endswith('?').sum()

np.int64(3)

np.int64(3)

np.int64(7)

np.int64(10)

In [203]:
# B7. Complaint flag

# (a) Make a True/False column complaint: text contains refund, delivery, broken or wrong (any case, na=False).
c_m['complaint']= c_m['text'].str.contains('refund|delivery|broken|wrong', case=False, na=False)
# (b) How many complaints?
(c_m['complaint']==True).sum()
# (c) Complaint rate per brand (groupby, mean, 2.6).
c_m.groupby('brand')['complaint'].mean().round(2).sort_values(ascending=False)
# (d) Round to 2.
# (e) Sort highest first.
# (f) Comment: which brand has the highest rate? Is UrbanThread’s 0.00 trustworthy? (Hint: how many UrbanThread comments are there?)
#FitFuel-- not trustworthy, it only has one comment, which has no complaint word flags
c_m[c_m['brand']=='UrbanThread']

np.int64(8)

brand
FitFuel        0.29
HomeNest       0.27
Glowly         0.15
SnackBox       0.12
UrbanThread    0.00
Name: complaint, dtype: float64

,comment_id,post_id,user,brand_raw,text,price_raw,brand,category,hq_country,followers,monthly_budget,text_len,complaint
1,C002,P0066,@hana,URBANTHREAD,LOVE the new flavour #snacktime,NaN,UrbanThread,fashion,US,45500.0,8000.0,31.0,False


In [204]:
# B8. Users

# (a) Make handle = user without the @. Do it two ways: .str.replace and .str[1:].
c_m['handle']= c_m['user'].str.replace("@", "")
c_m['handle']= c_m['user'].str[1:]
# (b) Top 3 handles by number of comments (value_counts().head(3)).
c_m['handle'].value_counts().head(3)

handle
hana      9
finn      5
gabi_x    4
Name: count, dtype: int64

In [205]:
# B9. Hashtags

# (a) Make n_hashtags = number of # in text.
c_m['n_hashtags']= c_m['text'].str.count('#')
# (b) Total hashtags across all comments.
c_m['n_hashtags'].sum()
# (c) Max hashtags in one comment.
c_m['n_hashtags'].max()

np.float64(6.0)

np.float64(2.0)

In [206]:
# B10. Prices: text → numbers

# (a) Show price_raw.unique().
c_m['price_raw'].unique()
# (b) Make price = price_raw with $, £ and , removed (three .str.replace steps), then .astype(float).
c_m['price']=c_m['price_raw'].str.replace(r'\$|£|,', "", regex=True).astype(float)
# (c) How many non-missing prices?
c_m['price'].notna().sum()
# (d) Median price.
c_m['price'].median()
# (e) Comment: some prices are $ and some are £. What’s wrong with calculating a single median of all of them?
# usd and gbp are diff currencies meaning 10 usd != 10 gbp, resulting in the wrong median

<ArrowStringArray>
['£8', nan, '$12.99', '$9.50', '£22.00', '$120', '$1,050.00', '$45']
Length: 8, dtype: str

np.int64(15)

np.float64(12.99)

In [207]:
c["brand_raw"].str.strip().str.lower() #.str missing

0          glowly
1     urbanthread
2         fitfuel
3        snackbox
4          glowly
5        homenest
6        snackbox
7          glowly
8         fitfuel
9          glowly
10         glowly
11      snack box
12         glowly
13       homenest
14       snackbox
15        fitfuel
16        fitfuel
17       homenest
18         glowly
19         glowly
20        fitfuel
21        fitfuel
22         glowly
23       homenest
24        fitfuel
25         glowly
26       homenest
27       homenest
28       homenest
29       snackbox
30       homenest
31       snackbox
32       snackbox
33         glowly
34       snackbox
35         glowly
36       homenest
37         glowly
38       homenest
39       homenest
Name: brand_raw, dtype: str

In [208]:
c["user"].str.replace("@", "") #will find a whole col cell , not within text

0      gabi_x
1        hana
2       ben.t
3      gabi_x
4        finn
5        hana
6        jo.l
7       ben.t
8     chloe99
9        hana
10      dev_m
11       hana
12      dev_m
13       finn
14    chloe99
15       hana
16    chloe99
17     gabi_x
18       finn
19     ivan_p
20       jo.l
21       jo.l
22     ivan_p
23       hana
24      ben.t
25    chloe99
26       hana
27      ben.t
28     ella.r
29     ella.r
30       hana
31     anna_k
32     gabi_x
33     anna_k
34       finn
35       hana
36       finn
37     anna_k
38      dev_m
39     ella.r
Name: user, dtype: str

In [209]:
c["price_raw"].str.replace(r'\$|£|,', "", regex=True).astype(float) #does not convert other currencies/special chars in text so they all remain as str 

0        8.00
1         NaN
2        8.00
3         NaN
4         NaN
5         NaN
6         NaN
7         NaN
8         NaN
9         NaN
10      12.99
11        NaN
12      12.99
13        NaN
14       9.50
15      22.00
16        NaN
17     120.00
18        NaN
19       9.50
20    1050.00
21        NaN
22        NaN
23        NaN
24        NaN
25        NaN
26       8.00
27        NaN
28    1050.00
29      45.00
30        NaN
31     120.00
32        NaN
33        NaN
34      12.99
35      12.99
36        NaN
37        NaN
38        NaN
39        NaN
Name: price_raw, dtype: float64